Producto Interno Bruto PIB - SERIES ANUALES

In [9]:

import requests
import pandas as pd
from Clave import APIKEY

indicadoresIDs=[120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174]
#indicadoresIDs=[131,174]

FechaInicio="2015-1-01T00:00:00"
FechaFinal="2015-1-01T00:00:00"
print(FechaInicio)
df_combinado=[]

for indicadorID in indicadoresIDs:
    urlCifras= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}/cifras"
    urlIndicadores= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}"
    
    params1={
        "fechainicio":FechaInicio
        ,"fechafinal":FechaFinal
    }
    
    headers = {
        "clave" : APIKEY,
        "Conten-Type": "application/json"
    }
    
    responseCifras = requests.get(urlCifras, headers=headers, params=params1)
    responseIndicadores = requests.get(urlIndicadores, headers=headers)
            
    if responseIndicadores.status_code == 200:
        datosIndicadores = responseIndicadores.json()
        #print(json.dumps(datosIndicadores,indent=4))  
        df_Indicadores=pd.DataFrame([datosIndicadores])
        ##print(df_Indicadores.to_string(index=False))
    else:
        print(f"Error {responseIndicadores.status_code}: {responseIndicadores.text}")
    
    if responseCifras.status_code == 200:
        datos1 = responseCifras.json()
        if datos1:
            df_Cifras=pd.DataFrame(datos1)
        else:
            df_Cifras=pd.DataFrame([{
            "Fecha":FechaInicio,
            "Id":0,       
            "IndicadorId":indicadorID,
            "Nombre":df_Indicadores.loc[0,"Nombre"],
            "Descripcion":df_Indicadores.loc[0,"Descripcion"],
            "Valor":0
            }])            
            ##print(df_Cifras.to_string(index=False))
    else:
        print(f"Error {responseCifras.status_code}: {responseCifras.text}")
  

    df_merge=pd.merge(
        df_Cifras,
        df_Indicadores,
        left_on='IndicadorId',
        right_on='Id',
        how='left'
    )

    #print(df_merge)
    #df_merge['Nombre_Indicador']= 'Indice de Precios al Consumidor IPC'  
    df_merge['NOMBRE_INDICADOR']=df_merge['Descripcion_x'].str.split('-').str[-1]
    df_merge['TIPO']=df_merge['Descripcion_x'].apply(lambda x:'-'.join(x.split(' - ')[1:2]))
    print(indicadorID, end=",")
    df_final=df_merge[['Fecha','Periodicidad','Valor','NOMBRE_INDICADOR','TIPO']]
    df_combinado.append(df_final)
else:
    print(f"Error {responseCifras.status_code}: {responseCifras.text}") 

df_resultado=pd.concat(df_combinado, ignore_index=True)
df_resultado=df_resultado.rename(columns={
    'Fecha':'FECHA',
    'IndicadorId':'INDICADOR_ID',
    'Periodicidad':'PERIODICIDAD',
    'Descripcion_x':'DESCRIPCION',
    'Valor':'VARIACION'
})
##df_merge.to_csv('indicador.csv', index=False, encoding='utf-8-sig')

2015-1-01T00:00:00
120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,Error 200: [{"Id":12450,"IndicadorId":174,"Nombre":"ESR-PIBA-01-77","Descripcion":"ESR-PIBA-01 - Corrientes - Variaci\u00F3n RAE - Transporte, Almacenamiento","Fecha":"2015-01-01T00:00:00","Valor":29.1000},{"Id":12618,"IndicadorId":174,"Nombre":"ESR-PIBA-01-77","Descripcion":"ESR-PIBA-01 - Corrientes - Variaci\u00F3n RAE - Transporte, Almacenamiento","Fecha":"2015-01-01T00:00:00","Valor":29.1000}]


In [11]:
df_resultado.head(10)

,FECHA,PERIODICIDAD,VARIACION,NOMBRE_INDICADOR,TIPO
0,2015-01-01T00:00:00,Anual,5.4,Actividades Inmobiliarias y Empresariales,Constantes
1,2015-01-01T00:00:00,Anual,5.4,Actividades Inmobiliarias y Empresariales,Constantes
2,2015-01-01T00:00:00,Anual,0.5,Administración Pública y Defensa; Planes de S...,Constantes
3,2015-01-01T00:00:00,Anual,0.5,Administración Pública y Defensa; Planes de S...,Constantes
4,2015-01-01T00:00:00,Anual,2.6,"Agricultura, Ganadería, Caza, Silvicultura y ...",Constantes
5,2015-01-01T00:00:00,Anual,2.6,"Agricultura, Ganadería, Caza, Silvicultura y ...",Constantes
6,2015-01-01T00:00:00,Anual,3.2,Comercio,Constantes
7,2015-01-01T00:00:00,Anual,3.2,Comercio,Constantes
8,2015-01-01T00:00:00,Anual,4.7,Comunicaciones,Constantes
9,2015-01-01T00:00:00,Anual,4.7,Comunicaciones,Constantes


In [12]:
#df_resultado.drop_duplicates().shape

df_resultado['NOMBRE_INDICADOR'] = df_resultado['NOMBRE_INDICADOR'].str.strip()
df_resultado['TIPO'] = df_resultado['TIPO'].str.strip()
df_resultado['TIPO'] = df_resultado['TIPO'] + ' - Rama de Actividad Económica'
df_resultado=df_resultado.drop_duplicates()
df_resultado.head(10)

,FECHA,PERIODICIDAD,VARIACION,NOMBRE_INDICADOR,TIPO
0,2015-01-01T00:00:00,Anual,5.4,Actividades Inmobiliarias y Empresariales,Constantes - Rama de Actividad Económica
2,2015-01-01T00:00:00,Anual,0.5,Administración Pública y Defensa; Planes de Se...,Constantes - Rama de Actividad Económica
4,2015-01-01T00:00:00,Anual,2.6,"Agricultura, Ganadería, Caza, Silvicultura y P...",Constantes - Rama de Actividad Económica
6,2015-01-01T00:00:00,Anual,3.2,Comercio,Constantes - Rama de Actividad Económica
8,2015-01-01T00:00:00,Anual,4.7,Comunicaciones,Constantes - Rama de Actividad Económica
10,2015-01-01T00:00:00,Anual,2.3,Construcción,Constantes - Rama de Actividad Económica
12,2015-01-01T00:00:00,Anual,8.8,Electricidad y Distribución de Agua,Constantes - Rama de Actividad Económica
14,2015-01-01T00:00:00,Anual,-1.0,Explotación de Minas y Canteras,Constantes - Rama de Actividad Económica
16,2015-01-01T00:00:00,Anual,2.8,Hoteles y Restaurantes,Constantes - Rama de Actividad Económica
18,2015-01-01T00:00:00,Anual,3.9,Industrias Manufactureras,Constantes - Rama de Actividad Económica


In [7]:
################           UPDATE TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from tqdm import tqdm
from sqlalchemy import create_engine, text
from Server import AZURE
credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'INDICADORES_MACROECONOMICOS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}
#columns_types

In [13]:


df_resultado['FECHA'] = pd.to_datetime(df_resultado['FECHA']).dt.strftime('%Y-%m-%d')
data_frame = df_resultado
chunksize = 100

with engine.begin() as connection:
    for start in tqdm(range(0, len(data_frame), chunksize), desc="Actualizando datos"):
        end = min(start + chunksize, len(data_frame))
        chunk = data_frame.iloc[start:end]

    for _, row in chunk.iterrows():
        try:
            update_query = text(f"""
                UPDATE {schema}.{tabla}
                SET VARIACION = :variacion
                WHERE FECHA = :fecha
                  AND PERIODICIDAD = :periodicidad
                  AND NOMBRE_INDICADOR = :nombre_indicador
                  AND TIPO = :tipo
            """)
            result = connection.execute(update_query, {
                'variacion': row['VARIACION'],
                'fecha': row['FECHA'],
                'periodicidad': row['PERIODICIDAD'],
                'nombre_indicador': row['NOMBRE_INDICADOR'],
                'tipo': row['TIPO']
            })
            print(f"Actualizando: {row['FECHA']}, {row['PERIODICIDAD']}, {row['NOMBRE_INDICADOR']} ,{row['TIPO']} -> Filas afectadas: {result.rowcount}")
        except Exception as e:
            print(f"Error al actualizar datos en la base de datos: {e}")

Actualizando datos: 100%|██████████| 1/1 [00:00<00:00, 6364.65it/s]


Actualizando: 2015-01-01, Anual, Actividades Inmobiliarias y Empresariales ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Administración Pública y Defensa; Planes de Seguridad Social de Afiliación Obligatoria ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Agricultura, Ganadería, Caza, Silvicultura y Pesca ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Comercio ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Comunicaciones ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Construcción ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Electricidad y Distribución de Agua ,Constantes - Rama de Actividad Económica -> Filas afectadas: 1
Actualizando: 2015-01-01, Anual, Explotación de Minas y Canteras ,Co

In [8]:
data_frame.head(10)


,FECHA,PERIODICIDAD,VARIACION,NOMBRE_INDICADOR,TIPO
0,2024-01-01,Anual,4.6,Actividades Inmobiliarias y Empresariales,Constantes - Rama de Actividad Económica - Ram...
1,2024-01-01,Anual,5.0,Administración Pública y Defensa; Planes de Se...,Constantes - Rama de Actividad Económica - Ram...
2,2024-01-01,Anual,-0.7,"Agricultura, Ganadería, Caza, Silvicultura y P...",Constantes - Rama de Actividad Económica - Ram...
3,2024-01-01,Anual,3.5,Comercio,Constantes - Rama de Actividad Económica - Ram...
4,2024-01-01,Anual,4.6,Comunicaciones,Constantes - Rama de Actividad Económica - Ram...
5,2024-01-01,Anual,5.2,Construcción,Constantes - Rama de Actividad Económica - Ram...
6,2024-01-01,Anual,11.5,Electricidad y Distribución de Agua,Constantes - Rama de Actividad Económica - Ram...
7,2024-01-01,Anual,1.7,Explotación de Minas y Canteras,Constantes - Rama de Actividad Económica - Ram...
8,2024-01-01,Anual,4.1,Hoteles y Restaurantes,Constantes - Rama de Actividad Económica - Ram...
9,2024-01-01,Anual,-2.0,Industrias Manufactureras,Constantes - Rama de Actividad Económica - Ram...
